In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/final/nifty_features_with_regime_5min.csv",
    low_memory=False)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

In [4]:
df = df.dropna(subset=[
    "ema_5",
    "ema_15",
    "market_regime",
    "open"
])

In [5]:
df["ema_cross_up"] = (
    (df["ema_5"] > df["ema_15"]) &
    (df["ema_5"].shift(1) <= df["ema_15"].shift(1))
)

df["ema_cross_down"] = (
    (df["ema_5"] < df["ema_15"]) &
    (df["ema_5"].shift(1) >= df["ema_15"].shift(1))
)

In [6]:
df["signal"] = 0          # 1 = long, -1 = short
df["position"] = 0

In [7]:
# LONG Entry
df.loc[
    (df["ema_cross_up"]) &
    (df["market_regime"] == 1),
    "signal"
] = 1

# SHORT Entry
df.loc[
    (df["ema_cross_down"]) &
    (df["market_regime"] == -1),
    "signal"
] = -1

In [8]:
df["position"] = df["signal"].shift(1)
df["position"] = df["position"].ffill().fillna(0)

In [9]:
# LONG Exit
df.loc[
    (df["position"] == 1) &
    (df["ema_cross_down"]),
    "position"
] = 0

# SHORT Exit
df.loc[
    (df["position"] == -1) &
    (df["ema_cross_up"]),
    "position"
] = 0

In [10]:
df["spot_returns"] = df["close"].pct_change()

df["strategy_returns"] = (
    df["position"].shift(1) * df["spot_returns"]
)

In [11]:
df["cum_market_returns"] = (1 + df["spot_returns"]).cumprod()
df["cum_strategy_returns"] = (1 + df["strategy_returns"]).cumprod()

In [12]:
df.to_csv(
    "../data/final/nifty_baseline_strategy_5min.csv",
    index=False
)

## Task 4.2: Backtesting & Metrics

In [14]:
split_idx = int(len(df) * 0.7)

train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

In [15]:
def sharpe_ratio(returns, freq=75):
    return np.sqrt(freq) * returns.mean() / returns.std()

def sortino_ratio(returns, freq=75):
    downside = returns[returns < 0]
    return np.sqrt(freq) * returns.mean() / downside.std()

def max_drawdown(cum_returns):
    peak = cum_returns.cummax()
    dd = (cum_returns - peak) / peak
    return dd.min()

def calmar_ratio(cum_returns, returns):
    total_return = cum_returns.iloc[-1] - 1
    mdd = abs(max_drawdown(cum_returns))
    return total_return / mdd if mdd != 0 else np.nan

In [16]:
def extract_trades(df):
    trades = []
    entry_time = None
    entry_price = None
    direction = None

    for i in range(1, len(df)):
        if df["position"].iloc[i] != df["position"].iloc[i-1]:
            # Entry
            if df["position"].iloc[i] != 0:
                entry_time = df["timestamp"].iloc[i]
                entry_price = df["open"].iloc[i]
                direction = df["position"].iloc[i]

            # Exit
            elif df["position"].iloc[i-1] != 0:
                exit_time = df["timestamp"].iloc[i]
                exit_price = df["open"].iloc[i]

                pnl = (
                    (exit_price - entry_price) / entry_price
                    if direction == 1
                    else (entry_price - exit_price) / entry_price
                )

                trades.append({
                    "entry_time": entry_time,
                    "exit_time": exit_time,
                    "pnl": pnl,
                    "duration": (exit_time - entry_time).seconds / 300
                })

    return pd.DataFrame(trades)

In [17]:
def backtest_metrics(df):
    returns = df["strategy_returns"].dropna()
    cum_returns = (1 + returns).cumprod()

    trades = extract_trades(df)

    metrics = {
        "Total Return": cum_returns.iloc[-1] - 1,
        "Sharpe Ratio": sharpe_ratio(returns),
        "Sortino Ratio": sortino_ratio(returns),
        "Calmar Ratio": calmar_ratio(cum_returns, returns),
        "Max Drawdown": max_drawdown(cum_returns),
        "Win Rate": (trades["pnl"] > 0).mean() if len(trades) > 0 else 0,
        "Profit Factor": (
            trades[trades["pnl"] > 0]["pnl"].sum() /
            abs(trades[trades["pnl"] < 0]["pnl"].sum())
            if len(trades[trades["pnl"] < 0]) > 0 else np.nan
        ),
        "Average Trade Duration (candles)": trades["duration"].mean() if len(trades) > 0 else 0,
        "Total Trades": len(trades)
    }

    return pd.Series(metrics)

In [18]:
train_metrics = backtest_metrics(train_df)
train_metrics

Total Return                         -1.000001
Sharpe Ratio                          0.035638
Sortino Ratio                         0.006084
Calmar Ratio                         -0.999999
Max Drawdown                         -1.000002
Win Rate                              0.455399
Profit Factor                         1.444009
Average Trade Duration (candles)      0.000000
Total Trades                        213.000000
dtype: float64

In [19]:
test_metrics = backtest_metrics(test_df)
test_metrics

Total Return                        -1.001504
Sharpe Ratio                        -0.005109
Sortino Ratio                       -0.000228
Calmar Ratio                        -0.720407
Max Drawdown                        -1.390193
Win Rate                             0.400000
Profit Factor                        0.956110
Average Trade Duration (candles)     0.000000
Total Trades                        55.000000
dtype: float64

In [20]:
metrics_df = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics
})

metrics_df.to_csv(
    "../data/final/baseline_strategy_metrics.csv"
)